In [1]:
import cv2
import numpy as np
from keras.models import load_model

I0000 00:00:1773534775.577495    1481 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1773534776.214010    1481 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1773534778.270254    1481 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


In [2]:
print("Loading model...")
model = load_model('models/binary_emotion_model.h5')

face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

# Keras alphabetical: irritated=0, neutral=1
# Sigmoid → close to 1.0 = neutral, close to 0.0 = irritated
THRESHOLD = 0.5

COLOR_NEUTRAL   = (50, 200, 100)  # green (BGR)
COLOR_IRRITATED = (50, 50, 220)   # red (BGR)
FONT = cv2.FONT_HERSHEY_SIMPLEX


def predict_face(gray_roi):
    """Predict emotion from grayscale face crop."""
    resized = cv2.resize(gray_roi, (48, 48))
    norm    = resized.astype('float32') / 255.0
    inp     = norm.reshape(1, 48, 48, 1)
    prob    = model.predict(inp, verbose=0)[0][0]
    
    if prob >= THRESHOLD:
        return 'NEUTRAL', prob, COLOR_NEUTRAL
    else:
        return 'IRRITATED', 1.0 - prob, COLOR_IRRITATED


def draw_confidence_bar(frame, prob_neutral, x, y, w):
    """Draw green/red confidence bar above face."""
    bar_y = max(y - 38, 0)
    bar_h = 14
    neutral_w = int(prob_neutral * w)
    
    # Background
    cv2.rectangle(frame, (x, bar_y), (x+w, bar_y+bar_h), (35,35,35), -1)
    
    # Neutral (green)
    if neutral_w > 0:
        cv2.rectangle(frame, (x, bar_y), (x+neutral_w, bar_y+bar_h), 
                     COLOR_NEUTRAL, -1)
    
    # Irritated (red)
    if neutral_w < w:
        cv2.rectangle(frame, (x+neutral_w, bar_y), (x+w, bar_y+bar_h), 
                     COLOR_IRRITATED, -1)
    
    # Percentage labels
    cv2.putText(frame,
        f"N:{prob_neutral*100:.0f}%  I:{(1-prob_neutral)*100:.0f}%",
        (x, bar_y - 5), FONT, 0.4, (220, 220, 220), 1
    )


Loading model...


I0000 00:00:1773534780.880186    1481 gpu_device.cc:2043] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1754 MB memory:  -> device: 0, name: NVIDIA GeForce RTX 3050 Ti Laptop GPU, pci bus id: 0000:01:00.0, compute capability: 8.6


In [3]:
cap = cv2.VideoCapture(0)
print("🎥 Starting webcam... Press 'Q' to quit")

while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    gray  = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(
        gray, scaleFactor=1.1, minNeighbors=5, minSize=(48, 48)
    )
    
    for (x, y, w, h) in faces:
        face_roi = gray[y:y+h, x:x+w]
        label, conf, color = predict_face(face_roi)
        
        # Face rectangle
        cv2.rectangle(frame, (x, y), (x+w, y+h), color, 3)
        
        # Label with background
        label_text = f"{label}  {conf*100:.1f}%"
        (tw, th), _ = cv2.getTextSize(label_text, FONT, 0.8, 2)
        lbl_y = y + h + 32 if y + h + 38 < frame.shape[0] else y - 12
        
        cv2.rectangle(frame, (x, lbl_y-th-9), (x+tw+12, lbl_y+6), color, -1)
        cv2.putText(frame, label_text, (x+6, lbl_y), FONT, 0.8, 
                   (255, 255, 255), 2)
        
        # Confidence bar
        prob_n = model.predict(
            (cv2.resize(face_roi, (48,48)).astype('float32')/255.0).reshape(1,48,48,1),
            verbose=0
        )[0][0]
        draw_confidence_bar(frame, prob_n, x, y, w)
    
    # Legend
    cv2.putText(frame, "[GREEN] NEUTRAL",  (10, 28), FONT, 0.6, COLOR_NEUTRAL,   2)
    cv2.putText(frame, "[RED] IRRITATED", (10, 56), FONT, 0.6, COLOR_IRRITATED, 2)
    
    cv2.imshow('Binary Emotion Detector - Press Q to quit', frame)
    
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
print("✅ Webcam detector stopped")

🎥 Starting webcam... Press 'Q' to quit
✅ Webcam detector stopped


[ WARN:0@18.187] global cap_v4l.cpp:1049 tryIoctl VIDEOIO(V4L2:/dev/video0): select() timeout.
